In [1]:
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import FancyBboxPatch

### Generate 1000 random sequences background

In [2]:
import random
import pandas as pd

def generate_random_dna_sequences(num_sequences=500, seq_length=400, seed=42):
    random.seed(seed)
    bases = ['A', 'T', 'C', 'G']  
    sequences = [''.join(random.choices(bases, k=seq_length)) for _ in range(num_sequences)]

    df = pd.DataFrame({
        'seq_id': [f"seq_{i+1}" for i in range(num_sequences)],
        'random_seq': sequences
    })
    
    return df

### Pick 100 sequences

In [3]:
df = generate_random_dna_sequences()
df

,seq_id,random_seq
0,seq_1,CATACCGATAACAACCACGAGCTAGTAAGCGCCGTCGCGCCAATAA...
1,seq_2,TATGGAACCTGGCAAAATGATCACTACAGGATAGCAAGTGCGCGGC...
2,seq_3,TTCGTACTCCTTACCAGACCGTCGGTACCGGTCTCTCATGTCCGGG...
3,seq_4,GACCTTTTACACGTCGGAAACGCAGTGGTGCCCGCAATTTACTTAG...
4,seq_5,GACCCGCGAACCCGTCTCTGTGCTAATTACTCCGGATCTTTAATTC...
...,...,...
495,seq_496,CCTTACCAATGATATTTAAATGATCCAAAGGCTGAGTGGGCGTAGT...
496,seq_497,GTATACTCGTGCTGCGCGTCACAATCTCATGTACCGAAGGTGAATA...
497,seq_498,TTCGACCGAAGTGGCCTAGTAGAGACGACTTGCCAGAACACTGTCT...
498,seq_499,AGCGACCTTGGACCGATCAGGACGATCTTAGCCTGTATCTGCGAGG...


### Place the 5 variants in the centre

In [4]:
motifs_df = pd.read_csv("/scratch/st-cdeboer-1/sambina/position_mpra/outputs/TFs_position/human/k562_master_regulator.csv")
motifs = motifs_df.set_index("Seed_motif")["Consensus"].to_dict()

mutated_motifs = {key + "_alt": "N" * len(value) for key, value in motifs.items()}

# Merge original and mutated motifs into one dictionary
mutated_motifs_list = {**motifs, **mutated_motifs}

print(mutated_motifs_list)
    

{'CTCFL_HUMAN.H11MO.0.A': 'GCCGCCAGGGGGCGCCG', 'EGR1_C2H2_1': 'TACGCCCACGCATT', 'EWSR1-FLI1_MA0149.1': 'GGAAGGAAGGAAGGAAGG', 'FOXO1_forkhead_2': 'GTAAACATGTTTAC', 'FOXC1_forkhead_1': 'AAGTAAATAAACA', 'FOXO1_forkhead_3': 'TTTCCCCACACG', 'GATA1+TAL1_MA0140.2': 'CTTATCTGTGAGGAGCAG', 'KLF15_HUMAN.H11MO.0.A': 'GGGGAGGAGCAGGGGGGGG', 'Arid3a_MA0151.1': 'ATTAAA', 'IRF5_IRF_1': 'CCGAAACCGAAACT', 'KLF13_C2H2_1': 'ATGCCACGCCCCTTTTTG', 'EGR1_HUMAN.H11MO.0.A': 'GAGGGCGTGGGCGGGGG', 'MYBL1_MA0776.1': 'ACCGTTAACGGT', 'MYBL1_MYB_4': 'GGCCGTTATAACCGTTA', 'MYBL1_MYB_1': 'ACCGTTAAACGG', 'MYBL1_MYB_3': 'AAAACCGTTAA', 'MYBA_MOUSE.H11MO.0.C': 'AGTGGCAGTTGGAGA', 'IKZF1_HUMAN.H11MO.0.C': 'TTGGGAGA', 'CTCFL_HUMAN.H11MO.0.A_alt': 'NNNNNNNNNNNNNNNNN', 'EGR1_C2H2_1_alt': 'NNNNNNNNNNNNNN', 'EWSR1-FLI1_MA0149.1_alt': 'NNNNNNNNNNNNNNNNNN', 'FOXO1_forkhead_2_alt': 'NNNNNNNNNNNNNN', 'FOXC1_forkhead_1_alt': 'NNNNNNNNNNNNN', 'FOXO1_forkhead_3_alt': 'NNNNNNNNNNNN', 'GATA1+TAL1_MA0140.2_alt': 'NNNNNNNNNNNNNNNNNN', 'KLF15_H

In [5]:
def place_motifs_with_offsets(df, motifs, offsets=[0]):
    new_data = []

    for _, row in df.iterrows():
        sequence = row["random_seq"]
        seq_id = row["seq_id"]
        seq_length = len(sequence)
        center_pos = seq_length // 2 

        for motif_name, motif_seq in motifs.items():
            motif_length = len(motif_seq)

            for offset in offsets:
                start_pos = center_pos - (motif_length // 2)
                
                if start_pos < 0 or start_pos + motif_length > seq_length:
                    continue  
                new_sequence = sequence[:start_pos] + motif_seq + sequence[start_pos + motif_length:]
                new_data.append([seq_id, motif_name, new_sequence, start_pos, offset])
     
    new_df = pd.DataFrame(new_data, columns=["seq_id", "motif_name", "seq_400", "start_pos", "offset"])
    return new_df


In [6]:
ref_df = place_motifs_with_offsets(df, mutated_motifs_list)

In [7]:
print(len(ref_df))
ref_df

18000


,seq_id,motif_name,seq_400,start_pos,offset
0,seq_1,CTCFL_HUMAN.H11MO.0.A,CATACCGATAACAACCACGAGCTAGTAAGCGCCGTCGCGCCAATAA...,192,0
1,seq_1,EGR1_C2H2_1,CATACCGATAACAACCACGAGCTAGTAAGCGCCGTCGCGCCAATAA...,193,0
2,seq_1,EWSR1-FLI1_MA0149.1,CATACCGATAACAACCACGAGCTAGTAAGCGCCGTCGCGCCAATAA...,191,0
3,seq_1,FOXO1_forkhead_2,CATACCGATAACAACCACGAGCTAGTAAGCGCCGTCGCGCCAATAA...,193,0
4,seq_1,FOXC1_forkhead_1,CATACCGATAACAACCACGAGCTAGTAAGCGCCGTCGCGCCAATAA...,194,0
...,...,...,...,...,...
17995,seq_500,MYBL1_MYB_4_alt,TAGGATCCCAGTAGCAGTGCAATATACCGTTCTACCGAGGGCTGGG...,192,0
17996,seq_500,MYBL1_MYB_1_alt,TAGGATCCCAGTAGCAGTGCAATATACCGTTCTACCGAGGGCTGGG...,194,0
17997,seq_500,MYBL1_MYB_3_alt,TAGGATCCCAGTAGCAGTGCAATATACCGTTCTACCGAGGGCTGGG...,195,0
17998,seq_500,MYBA_MOUSE.H11MO.0.C_alt,TAGGATCCCAGTAGCAGTGCAATATACCGTTCTACCGAGGGCTGGG...,193,0


### Create the 200 bp sequences with the offsets

In [8]:
import pandas as pd

# Define colors
BLUE = "\033[1;34m"  # Blue for 80 bp window
RED = "\033[1;31m"   # Red for motif
RESET = "\033[0m"    # Reset color

def color_text(sequence, motif_start, motif_length, window_start, window_end):
    """
    Colors the sequence:
    - Blue: The extracted 80 bp window.
    - Red: The motif inside the extracted window.
    """
    colored_sequence = ""

    for i, base in enumerate(sequence):
        if window_start <= i < window_end:  # Inside 80 bp window
            if motif_start <= i < motif_start + motif_length:  # Motif inside the 80 bp window
                colored_sequence += f"{RED}{base}{RESET}"  # Red for motif
            else:
                colored_sequence += f"{BLUE}{base}{RESET}"  # Blue for 80 bp window
        else:
            colored_sequence += base  # Default color (no highlight)

    return colored_sequence

def extract_80bp_windows_with_offsets(df, allele, motif_col="motif_name", offsets=range(-80, 81)):
    """
    Extracts 80 bp windows from sequences, centering the motif at various offsets.

    Parameters:
    - df: DataFrame containing sequences and motifs.
    - allele: "ref" or "alt" to select sequence column.
    - motif_col: Column with motif names.
    - offsets: List of offsets to shift the motif.

    Returns:
    - DataFrame with extracted 200 bp windows for each offset.
    """
    seq_col = f"seq_400"

    for offset in offsets:
        extracted_windows = []

        for _, row in df.iterrows():
            sequence = row[seq_col]  # Full 200 bp sequence
            motif_name = row[motif_col]

            # Get motif sequence
            if motif_name in mutated_motifs_list:
                motif = mutated_motifs_list[motif_name]
            else:
                raise ValueError(f"Motif '{motif_name}' not found in the motifs dictionary.")

            motif_length = len(motif)
            seq_length = len(sequence)  
            center = seq_length // 2  

            motif_start = center - (motif_length // 2) + offset  

            window_start = motif_start - (100 - motif_length // 2)
            window_end = window_start + 200
            extracted_window = sequence[window_start:window_end]
            extracted_windows.append(extracted_window)

        df[f"seq_{offset}"] = extracted_windows

    return df

# Run on reference DataFrame
df_ref_80bp = extract_80bp_windows_with_offsets(ref_df, "ref")
df_ref_80bp.head(5)


/tmp/ipykernel_729375/2473980462.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"seq_{offset}"] = extracted_windows
/tmp/ipykernel_729375/2473980462.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"seq_{offset}"] = extracted_windows
/tmp/ipykernel_729375/2473980462.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented fr

,seq_id,motif_name,seq_400,start_pos,offset,seq_-80,seq_-79,seq_-78,seq_-77,seq_-76,...,seq_71,seq_72,seq_73,seq_74,seq_75,seq_76,seq_77,seq_78,seq_79,seq_80
0,seq_1,CTCFL_HUMAN.H11MO.0.A,CATACCGATAACAACCACGAGCTAGTAAGCGCCGTCGCGCCAATAA...,192,0,GCTAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGG...,CTAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGA...,TAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAA...,AGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAAT...,GTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAATT...,...,CTGGAATTTCCGATTGAATTTGCCGCCAGGGGGCGCCGGTGTTGGC...,TGGAATTTCCGATTGAATTTGCCGCCAGGGGGCGCCGGTGTTGGCC...,GGAATTTCCGATTGAATTTGCCGCCAGGGGGCGCCGGTGTTGGCCA...,GAATTTCCGATTGAATTTGCCGCCAGGGGGCGCCGGTGTTGGCCAT...,AATTTCCGATTGAATTTGCCGCCAGGGGGCGCCGGTGTTGGCCATG...,ATTTCCGATTGAATTTGCCGCCAGGGGGCGCCGGTGTTGGCCATGC...,TTTCCGATTGAATTTGCCGCCAGGGGGCGCCGGTGTTGGCCATGCC...,TTCCGATTGAATTTGCCGCCAGGGGGCGCCGGTGTTGGCCATGCCC...,TCCGATTGAATTTGCCGCCAGGGGGCGCCGGTGTTGGCCATGCCCA...,CCGATTGAATTTGCCGCCAGGGGGCGCCGGTGTTGGCCATGCCCAC...
1,seq_1,EGR1_C2H2_1,CATACCGATAACAACCACGAGCTAGTAAGCGCCGTCGCGCCAATAA...,193,0,GCTAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGG...,CTAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGA...,TAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAA...,AGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAAT...,GTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAATT...,...,CTGGAATTTCCGATTGAATTTATACGCCCACGCATTATGTGTTGGC...,TGGAATTTCCGATTGAATTTATACGCCCACGCATTATGTGTTGGCC...,GGAATTTCCGATTGAATTTATACGCCCACGCATTATGTGTTGGCCA...,GAATTTCCGATTGAATTTATACGCCCACGCATTATGTGTTGGCCAT...,AATTTCCGATTGAATTTATACGCCCACGCATTATGTGTTGGCCATG...,ATTTCCGATTGAATTTATACGCCCACGCATTATGTGTTGGCCATGC...,TTTCCGATTGAATTTATACGCCCACGCATTATGTGTTGGCCATGCC...,TTCCGATTGAATTTATACGCCCACGCATTATGTGTTGGCCATGCCC...,TCCGATTGAATTTATACGCCCACGCATTATGTGTTGGCCATGCCCA...,CCGATTGAATTTATACGCCCACGCATTATGTGTTGGCCATGCCCAC...
2,seq_1,EWSR1-FLI1_MA0149.1,CATACCGATAACAACCACGAGCTAGTAAGCGCCGTCGCGCCAATAA...,191,0,GCTAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGG...,CTAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGA...,TAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAA...,AGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAAT...,GTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAATT...,...,CTGGAATTTCCGATTGAATTGGAAGGAAGGAAGGAAGGGTGTTGGC...,TGGAATTTCCGATTGAATTGGAAGGAAGGAAGGAAGGGTGTTGGCC...,GGAATTTCCGATTGAATTGGAAGGAAGGAAGGAAGGGTGTTGGCCA...,GAATTTCCGATTGAATTGGAAGGAAGGAAGGAAGGGTGTTGGCCAT...,AATTTCCGATTGAATTGGAAGGAAGGAAGGAAGGGTGTTGGCCATG...,ATTTCCGATTGAATTGGAAGGAAGGAAGGAAGGGTGTTGGCCATGC...,TTTCCGATTGAATTGGAAGGAAGGAAGGAAGGGTGTTGGCCATGCC...,TTCCGATTGAATTGGAAGGAAGGAAGGAAGGGTGTTGGCCATGCCC...,TCCGATTGAATTGGAAGGAAGGAAGGAAGGGTGTTGGCCATGCCCA...,CCGATTGAATTGGAAGGAAGGAAGGAAGGGTGTTGGCCATGCCCAC...
3,seq_1,FOXO1_forkhead_2,CATACCGATAACAACCACGAGCTAGTAAGCGCCGTCGCGCCAATAA...,193,0,GCTAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGG...,CTAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGA...,TAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAA...,AGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAAT...,GTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAATT...,...,CTGGAATTTCCGATTGAATTTAGTAAACATGTTTACATGTGTTGGC...,TGGAATTTCCGATTGAATTTAGTAAACATGTTTACATGTGTTGGCC...,GGAATTTCCGATTGAATTTAGTAAACATGTTTACATGTGTTGGCCA...,GAATTTCCGATTGAATTTAGTAAACATGTTTACATGTGTTGGCCAT...,AATTTCCGATTGAATTTAGTAAACATGTTTACATGTGTTGGCCATG...,ATTTCCGATTGAATTTAGTAAACATGTTTACATGTGTTGGCCATGC...,TTTCCGATTGAATTTAGTAAACATGTTTACATGTGTTGGCCATGCC...,TTCCGATTGAATTTAGTAAACATGTTTACATGTGTTGGCCATGCCC...,TCCGATTGAATTTAGTAAACATGTTTACATGTGTTGGCCATGCCCA...,CCGATTGAATTTAGTAAACATGTTTACATGTGTTGGCCATGCCCAC...
4,seq_1,FOXC1_forkhead_1,CATACCGATAACAACCACGAGCTAGTAAGCGCCGTCGCGCCAATAA...,194,0,GCTAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGG...,CTAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGA...,TAGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAA...,AGTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAAT...,GTAAGCGCCGTCGCGCCAATAAATCTTATGCCACATGCCCGGAATT...,...,CTGGAATTTCCGATTGAATTTAGAAGTAAATAAACAATGTGTTGGC...,TGGAATTTCCGATTGAATTTAGAAGTAAATAAACAATGTGTTGGCC...,GGAATTTCCGATTGAATTTAGAAGTAAATAAACAATGTGTTGGCCA...,GAATTTCCGATTGA

In [9]:
def validate_sequence_lengths(df, allele, expected_length, offsets=range(-80, 81)):
    """
    Checks if sequences in the DataFrame have the expected length (80 bp) for given offsets.

    Parameters:
    - df: DataFrame containing sequences.
    - allele: "ref" or "alt" to check the correct sequence column.
    - offsets: List of offsets to check.
    - expected_length: Expected length of sequences (default: 80 bp).

    Raises:
    - ValueError if any sequence length is not 80 bp.
    """
    for offset in offsets:
        column_name = f"seq_{offset}"  # Column name format
        
        if column_name not in df.columns:
            raise ValueError(f"Column '{column_name}' not found in DataFrame.")

        # Check sequence lengths
        for i, seq in enumerate(df[column_name]):
            if len(seq) != expected_length:
                raise ValueError(f"Error in row {i}, column '{column_name}': Expected {expected_length} bp, got {len(seq)} bp")

    print("✅ All sequences have the correct length.")

validate_sequence_lengths(df_ref_80bp, "ref", 200)


✅ All sequences have the correct length.


In [10]:
test = pd.read_csv("/scratch/st-cdeboer-1/sambina/mpra/mpra_with_chromosome/agarwal/data_k562/fold_0/valid.txt", sep="\t")
left_adapter = test.loc[0, "seq"][:15]
right_adapter = test.loc[0, "seq"][-15:]
print(left_adapter)
print(right_adapter)

for col in df_ref_80bp.columns:
    if col.startswith('seq_'):  
        if col != "seq_id":
            df_ref_80bp[col] = df_ref_80bp[col].apply(lambda seq: left_adapter + seq + right_adapter)

validate_sequence_lengths(df_ref_80bp, "ref", 230)


AGGACCGGATCAACT
CATTGCGTGAACCGA
✅ All sequences have the correct length.


In [11]:
df_ref_80bp.to_csv("/scratch/st-cdeboer-1/sambina/position_mpra/outputs/TFs_position/human/k562_regulator_knockout_agarwal.csv")

### Create Reverse Complement

In [12]:
from Bio.Seq import Seq

def process_sequence(seq):
    if seq.startswith(left_adapter):
        seq = seq[len(left_adapter):]  
    if seq.endswith(right_adapter):
        seq = seq[:-len(right_adapter)]  
    
    rev_comp_seq = str(Seq(seq).reverse_complement())

    return left_adapter + rev_comp_seq + right_adapter

df_80bp_rc = df_ref_80bp.copy()  
for col in df_80bp_rc.columns:
    if col.startswith("seq_"):
        if col!="seq_id":
            df_80bp_rc[col] = df_80bp_rc[col].apply(process_sequence)

df_80bp_rc.to_csv("/scratch/st-cdeboer-1/sambina/position_mpra/outputs/TFs_position/human/k562_regulator_knockout_agarwal_rc.csv")


In [13]:
df_80bp_rc.shape

(18000, 166)